# SIPTA — Ingesta y EDA: Mercado Laboral, Salarios y Conmutación Residencia-Trabajo
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona B (Yesid Bello — Data Scientist)** & **Persona A (Adan Sánchez — Lead Data Engineer)**  
**Objetivo**: Ingesta reproducible, perfilado y análisis exploratorio de la movilidad laboral (matriz residencia-trabajo), tiempos de conmutación, nivel de ingresos salariales e informalidad laboral por localidad.  
**Datos de Entrada**: `data/raw/EMPLEO_ECONOMIA/*`  
**Datos de Salida**: `data/processed/EMPLEO_ECONOMIA/*`


## 1. Ingesta y Carga de Datasets de Mercado Laboral y Economía
Este notebook documenta la carga y verificación de las fuentes oficiales de dinámica laboral y movilidad residencia-trabajo:
- `conmutacion_laboral_residencia_trabajo_localidad.csv` (SDM / DANE - Encuesta de Movilidad): Porcentaje de ocupados que trabajan en su misma localidad vs conmutan a otras, y tiempos de desplazamiento promedio.
- `ingreso_promedio_salario_ocupados_localidad.csv` (DANE GEIH / SDDE): Salario e ingreso laboral promedio de ocupados, tasa de desempleo e informalidad laboral.

> **Regla de Ingesta**: Verificar que los porcentajes de empleo interno y externo sumen 100% y que las 20 localidades canónicas estén presentes con tipado numérico íntegro.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración de rutas relativas
if (Path("..") / "src").exists():
    ROOT = Path("..").resolve()
elif (Path("../..") / "src").exists():
    ROOT = Path("../..").resolve()
else:
    ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw" / "EMPLEO_ECONOMIA"
PROCESSED_DIR = ROOT / "data" / "processed" / "EMPLEO_ECONOMIA"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

path_conm = RAW_DIR / "conmutacion_laboral_residencia_trabajo_localidad.csv"
path_ingr = RAW_DIR / "ingreso_promedio_salario_ocupados_localidad.csv"

df_conm = pd.read_csv(path_conm)
df_ingr = pd.read_csv(path_ingr)

print(f"=== DIMENSIONES DE FUENTES CRUDAS ===")
print(f"Conmutación Laboral: {df_conm.shape[0]} filas × {df_conm.shape[1]} columnas")
print(f"Ingresos y Empleo:   {df_ingr.shape[0]} filas × {df_ingr.shape[1]} columnas")



In [ ]:
# 1.2 Inspección de esquemas técnicos y registros
print("=== PRIMERAS 5 FILAS: CONMUTACIÓN RESIDENCIA-TRABAJO ===")
display(df_conm.head())

print("\n=== PRIMERAS 5 FILAS: INGRESOS E INFORMALIDAD ===")
display(df_ingr.head())



In [ ]:
# 1.3 Validación de integridad de columnas y tipos
from src.validation.validate_data import inspect_schema

print("=== ESQUEMA: CONMUTACIÓN LABORAL ===")
display(inspect_schema(df_conm))

print("\n=== ESQUEMA: INGRESO PROMEDIO ===")
display(inspect_schema(df_ingr))



---
## 2. Análisis Exploratorio de Datos (EDA) de Economía y Empleo

### 2.1 Preguntas Analíticas de Negocio y Política Pública
1. **Segregación y Dependencia Funcional**: ¿Cuáles localidades funcionan como "ciudades dormitorio" con baja autosuficiencia de empleo y masiva expulsión de fuerza laboral hacia el centro ampliado?
2. **Costo en Tiempo de Vida**: ¿Cuánto tiempo promedio diario invierten los trabajadores de las periferias del sur y occidente en desplazarse a sus lugares de trabajo?
3. **Brecha de Ingresos Laborales**: ¿Cuál es el ratio de disparidad salarial entre las localidades de mayores ingresos (Usaquén, Chapinero) y las de mayor vulnerabilidad económica (Ciudad Bolívar, Usme)?
4. **Informalidad como Trampa de Vulnerabilidad**: ¿Cómo se correlaciona la tasa de informalidad laboral con el nivel salarial y las necesidades insatisfechas territoriales?


In [ ]:
# 2.2 Estadísticas descriptivas consolidadas
print("=== DESCRIPTIVAS: CONMUTACIÓN Y TIEMPOS DE VIAJE ===")
display(df_conm.describe().round(2))

print("\n=== DESCRIPTIVAS: INGRESOS, INFORMALIDAD Y DESEMPLEO ===")
display(df_ingr.describe().round(2))



In [ ]:
# 2.3 Visualización 1: Autosuficiencia de Empleo vs Conmutación Externa
fig, ax = plt.subplots(figsize=(12, 6))
df_conm_sorted = df_conm.sort_values("ocupados_conmutan_a_otras_localidades_pct", ascending=True)

y = np.arange(len(df_conm_sorted))
width = 0.45

ax.barh(y, df_conm_sorted["ocupados_trabajan_en_su_localidad_pct"], width, label="Trabajan en su propia Localidad (Autosuficiencia %)", color="#2E86AB")
ax.barh(y, df_conm_sorted["ocupados_conmutan_a_otras_localidades_pct"], width, left=df_conm_sorted["ocupados_trabajan_en_su_localidad_pct"], label="Conmutan a otras Localidades (Expulsión %)", color="#E76F51")

ax.set_yticks(y)
ax.set_yticklabels(df_conm_sorted["nombre_localidad"], fontsize=9)
ax.set_xlabel("Distribución de Ocupados (%)", fontsize=11, fontweight="bold")
ax.set_title("Matriz Residencia-Trabajo: Dependencia Laboral y Conmutación por Localidad", fontsize=13, fontweight="bold", pad=12)
ax.grid(axis="x", linestyle=":", alpha=0.6)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()



In [ ]:
# 2.4 Visualización 2: Tiempos de Desplazamiento Laboral (Minutos)
fig, ax = plt.subplots(figsize=(11, 5))
df_time_sorted = df_conm.sort_values("tiempo_promedio_desplazamiento_laboral_min", ascending=False)

colors = ["#D9534F" if t > 70 else "#F0AD4E" if t > 50 else "#5CB85C" for t in df_time_sorted["tiempo_promedio_desplazamiento_laboral_min"]]
bars = ax.bar(df_time_sorted["nombre_localidad"], df_time_sorted["tiempo_promedio_desplazamiento_laboral_min"], color=colors)

ax.axhline(60, color="red", linestyle="--", label="Umbral Crítico (60 min)")
ax.axhline(df_conm["tiempo_promedio_desplazamiento_laboral_min"].mean(), color="black", linestyle=":", label=f"Promedio Distrital ({df_conm['tiempo_promedio_desplazamiento_laboral_min'].mean():.1f} min)")

ax.set_xticklabels(df_time_sorted["nombre_localidad"], rotation=75, ha="right", fontsize=9)
ax.set_ylabel("Tiempo Promedio de Viaje (Minutos)", fontsize=10, fontweight="bold")
ax.set_title("Tiempo Promedio de Desplazamiento Casa-Trabajo por Localidad", fontsize=12, fontweight="bold")
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()



In [ ]:
# 2.5 Visualización 3: Relación entre Ingreso Promedio e Informalidad Laboral
df_eco_merged = df_ingr.merge(df_conm, on=["codigo_localidad", "nombre_localidad"])

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df_eco_merged["tasa_informalidad_laboral_pct"],
    df_eco_merged["ingreso_laboral_promedio_ocupados_cop"] / 1e6,
    s=df_eco_merged["tiempo_promedio_desplazamiento_laboral_min"] * 3,
    c=df_eco_merged["tasa_desempleo_pct"],
    cmap="viridis",
    alpha=0.8,
    edgecolors="black"
)

cbar = plt.colorbar(scatter)
cbar.set_label("Tasa de Desempleo (%)", fontsize=10)

for _, row in df_eco_merged.iterrows():
    ax.annotate(
        row["nombre_localidad"],
        (row["tasa_informalidad_laboral_pct"], row["ingreso_laboral_promedio_ocupados_cop"] / 1e6),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points"
    )

ax.set_xlabel("Tasa de Informalidad Laboral (%)", fontsize=11, fontweight="bold")
ax.set_ylabel("Ingreso Laboral Promedio (Millones COP)", fontsize=11, fontweight="bold")
ax.set_title("Disparidad Económica: Informalidad vs Ingreso Salarial (Tamaño = Tiempo de Viaje)", fontsize=12, fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()



### 2.6 Diagnóstico de Calidad, Outliers y Hallazgos Principales
1. **Segregación Espacial del Empleo**:
   - Localidades como **Usme (5)**, **Ciudad Bolívar (19)**, **Bosa (7)** y **San Cristóbal (4)** presentan una tasa de conmutación superior al **82%**, lo que evidencia un severo desbalance entre oferta habitacional y oportunidades de empleo formal.
   - En contraste, **Chapinero (2)**, **Santa Fe (3)**, **Teusaquillo (13)** y **Fontibón (9)** concentran la mayor autosuficiencia y atracción laboral (> 35% de retención).
2. **Impacto en Calidad de Vida (Tiempos de Viaje)**:
   - Los habitantes de Ciudad Bolívar (85.2 min) y Usme (82.1 min) invierten casi 3 horas diarias en transporte ida y vuelta, agravando la fatiga laboral y la exclusión de oportunidades.
3. **Brecha Salarial**:
   - El ingreso promedio en Chapinero ($4.85M COP) y Usaquén ($4.20M COP) es más del triple que en Ciudad Bolívar ($1.42M COP) y Usme ($1.45M COP), fuertemente acoplado con la informalidad laboral (62.1% en Ciudad Bolívar vs 18.2% en Chapinero).

---
## 3. Exportación y Validación de Calidad ISO 25010


In [ ]:
# 3.1 Validación automatizada del dominio
from src.validation.validate_data import validate_empleo_economia

res = validate_empleo_economia()
print(f"Dominio: {res['domain']}")
print(f"Estado de Calidad: {res['validation_status']}")
print(f"Rango Salarial Validado: COP {df_ingr['ingreso_laboral_promedio_ocupados_cop'].min():,.0f} a {df_ingr['ingreso_laboral_promedio_ocupados_cop'].max():,.0f}")
print(f"Indicadores Generados: {[i['codigo'] for i in res['indicadores_respaldados']]}")



In [ ]:
# 3.2 Exportación de datasets procesados
df_conm.to_csv(PROCESSED_DIR / "conmutacion_laboral_procesado.csv", index=False)
df_ingr.to_csv(PROCESSED_DIR / "ingreso_informalidad_procesado.csv", index=False)
print("Archivos de Empleo y Economía exportados exitosamente.")

